# Thực nghiệm cuối kỳ T2f
Thêm t2f-final.zip vào Kaggle Input hoặc chọn ZIP trên Colab. Có thể chạy trực tiếp khi notebook nằm trong repo đã giải nén.

In [ ]:
from pathlib import Path
import subprocess, os, sys, zipfile, shutil
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo = next((p for p in (Path.cwd(), Path.cwd().parent, work/'t2f-forecasting') if (p/'final/compare.py').is_file()), None)
if repo is None:
    repo = next((p.parent.parent for p in Path('/kaggle/input').rglob('final/compare.py')),None)
if repo is None:
    path=next((p for base in [work,Path('/kaggle/input')] for p in base.rglob('t2f-final.zip')),None)
    if path is None and 'google.colab' in sys.modules:
        from google.colab import files
        path=Path(next(iter(files.upload())))
    if path is None: raise FileNotFoundError('Thêm t2f-final.zip vào notebook')
    with zipfile.ZipFile(path) as z:z.extractall(work)
    repo=work/'t2f-forecasting'
print(repo)

In [ ]:
datasets=['ETTh','ETTm','ELD']
bridge=True
long_horizon=True
run_td3=True
run_patch=True
audit=False
output=work/'final_runs'
env={**os.environ,'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1'}
def run(module,*args):
    subprocess.run([sys.executable,'-m','final.'+module,*args,'--output',str(output)],cwd=repo,env=env,check=True)
for ds in datasets:
    run('compare','--dataset',ds)
    if bridge: run('compare','--dataset',ds,'--bridge')
    if run_td3: run('td3','--dataset',ds)
    if ds!='ELD':
        if run_patch: run('patch_transformer','--dataset',ds)
        if long_horizon:
            run('compare','--dataset',ds,'--horizon','96')
            if run_patch: run('patch_transformer','--dataset',ds,'--horizon','96')
    if audit:
        subprocess.run([sys.executable,'-m','final.local_audit','--dataset',ds if ds!='ELD' else 'eld','--output',str(output/'local_audit')],cwd=repo,env=env,check=True)
run('summarize')
print(shutil.make_archive(str(work/'final_results'),'zip',output))